In [1]:
import project_path
from rsatools import *

In [2]:
rng = prime_rng(8675_309)

p = rng.randprime(2**31, 2**32)
q = rng.randprime(2**31, 2**32)

n = p*q

lamb = carmichael(p, q)

e = 257
d = inverse257(lamb)

secrets = [p, q, n, lamb, e, d]

for num in secrets:
    bitlength = len(str(bin(num)))-2
    print(f"{bitlength:2.0f}    {num}")

32    3721615301
32    2961235187
64    11020578181798796287
59    423868391350613300
 9    257
58    222654602460438893


In [3]:
m_plaintext = "Hello, W"

m_encoding = int(codecs.encode(m_plaintext, encoding="ascii_emoji").hex(), 16)

c_encoding = pow(m_encoding, e, mod=n)

c_plaintext = codecs.decode(bytes.fromhex(hex(c_encoding)[2:]), encoding="ascii_emoji")

r_encoding = pow(c_encoding, d, mod=n)

r_plaintext = codecs.decode(bytes.fromhex(hex(r_encoding)[2:]), encoding="ascii_emoji")

print(m_plaintext)
print(c_plaintext)
print(r_plaintext)

Hello, W
(a⭐lKhW3
Hello, W


fucking finally

# To Do:

- automatic padding

- break long messages into words (is this necessary?)

- make it distributable

- increase the bounds for randprime...

okay let's test the long message thing first

In [4]:
p2 = rng.randprime(2**63, 2**64)
q2 = rng.randprime(2**63, 2**64)

n2 = p2*q2

lamb2 = carmichael(p2, q2)

e = 257
d2 = inverse257(lamb2)

secrets2 = [p2, q2, n2, lamb2, e, d2]

for num in secrets2:
    bitlength2 = len(str(bin(num)))-2
    print(f"{bitlength2:3.0f}    {num}")

 64    16735907832055708781
 64    14995498527984231137
128    250963281260071146001697860887864513997
123    5703710937728889681135601239268740320
  9    257
121    1908634788500717947773002749327282753


In [7]:
m_plaintext2 = "Hello, World!"
m_encoding2 = int(codecs.encode(m_plaintext2, encoding="ascii_emoji").hex(), 16)
c_encoding2 = pow(m_encoding2, e, mod=n2)
r_encoding2 = pow(c_encoding2, d2, mod=n2)
r_plaintext2 = codecs.decode(bytes.fromhex(hex(r_encoding2)[2:]), encoding="ascii_emoji")

print(r_plaintext2)

c_plaintext2 = codecs.decode(bytes.fromhex(hex(c_encoding2)[2:]), encoding="ascii_emoji")

Hello, World!


KeyError: b'\xfd'

Okay, I thought we might have gotten past this bug, but I guess not.  I knew it was going to rear its ugly head at some point.  Now, codecs probably has some way of dealing with this exact sort of error- but statistically, about half of the stuff in the ciphertext will encounter this bug.  so dealing with it by putting in a blank space, or a ?, or something like that, will make the ciphertext really boring and ugly.  so instead I'm going to fake it.